**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/mah-2023"], check=True)
    os.chdir("/content/encore/entries/mah-2023")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Earlier findings of mah-2023-01, checked on DANDI:001169 version 0.261120.1450

Attempt mah-2023-01 tested Mah et al. (2023) on version 0.260825.1909 of dandiset 001169. Attempt mah-2023-02, made blind to it, tested version 0.261120.1450. This notebook belongs to the reconciliation of the two (docs/protocol.md, "Repeat attempts"). It checks, on version 0.261120.1450, each finding of mah-2023-01 that no finding of mah-2023-02 reports: findings 3, 4, 5, 6, 9, 10 and 12. Findings 1, 2, 7, 8 and 11 are reported again by mah-2023-02 and are not checked here.

Each section opens with the earlier finding's statement. The data findings are checked on every file of the version; for findings 3 and 6 the earlier version is read too, to show what it held. The findings about the paper's code are checked against the code at the commit the claims file pins (github.com/constantinoplelab/published at b7b68ca8c107).

In [ ]:
import io, re, time, urllib.request
from concurrent.futures import ThreadPoolExecutor
import h5py
import numpy as np
import pandas as pd
import mah2023 as M

EARLIER_VERSION = "0.260825.1909"
CODE = "https://raw.githubusercontent.com/constantinoplelab/published/b7b68ca8c107/rat_behavior/"
T0 = time.time()


def read_summary(asset):
    """What the checks need from one file: column names, the two is_catch flags and their
    descriptions, the subject's age fields, the center_port_cue argument, and the mixed block
    catch trial arrays of finding 12."""
    b = M._get(f"{M.API}/assets/{asset['asset_id']}/download/")
    with h5py.File(io.BytesIO(b), "r") as f:
        p, r, s = f["intervals/processed_trials"], f["intervals/trials"], f["general/subject"]
        pc, rc = p["is_catch"][:].astype(bool), r["is_catch"][:].astype(bool)
        ta = f["general/task/task_arguments"]
        args = {n.decode(): e.decode() for n, e in zip(ta["argument_name"][:], ta["expression"][:])}
        out = dict(
            path=asset["path"],
            subject=s["subject_id"][()].decode(),
            date=re.search(r"-(\d{8})-", asset["path"]).group(1),
            proc_cols=tuple(sorted(p.keys())),
            n_trials=len(pc),
            catch_differ=int((pc != rc).sum()) if len(pc) == len(rc) else -1,
            proc_catch_desc=p["is_catch"].attrs.get("description", ""),
            raw_catch_desc=r["is_catch"].attrs.get("description", ""),
            age=s["age"][()].decode() if "age" in s else "",
            date_of_birth=s["date_of_birth"][()].decode() if "date_of_birth" in s else "",
            center_port_cue=args.get("center_port_cue", "absent"),
        )
        arrays = dict(wait_time=p["wait_time"][:], is_catch=pc, is_opt_out=p["is_opt_out"][:].astype(bool),
                      block=p["block_type"][:], vol=p["reward_volume_ul"][:])
    return out, arrays


def read_subject(asset):
    """The subject group of one file."""
    b = M._get(f"{M.API}/assets/{asset['asset_id']}/download/")
    with h5py.File(io.BytesIO(b), "r") as f:
        s = f["general/subject"]
        return dict(subject=s["subject_id"][()].decode(),
                    age=s["age"][()].decode() if "age" in s else "",
                    date_of_birth=s["date_of_birth"][()].decode() if "date_of_birth" in s else "")


def list_version(version, path=None):
    url = f"{M.API}/dandisets/{M.DANDISET}/versions/{version}/assets/?page_size=1000"
    if path:
        url += f"&path={path}"
    out = []
    while url:
        r = M.json.loads(M._get(url))
        out += r["results"]
        url = r["next"]
    return out

In [ ]:
assets = M.list_assets()
with ThreadPoolExecutor(24) as ex:
    res = list(ex.map(read_summary, assets))
files = pd.DataFrame([o for o, _ in res])
print(f"read {len(files)} files in {time.time() - T0:.0f} s")

In [ ]:
print("Dandiset", M.DANDISET, "version", M.VERSION, "| assets:", len(assets), "| files read:", len(files))
print("rats:", files.subject.nunique(), "| trials:", files.n_trials.sum())
print("dates:", files.date.min(), "to", files.date.max())

## Finding 3

Earlier statement: "One session is deposited twice, sub-L011_ses-RWTautowait2-20221004-095612.nwb and the same name with a "-x" suffix, with identical trial data."

The cell lists the files of that session in both versions, and looks for any other file name or session (rat and start time) that occurs twice in this version.

In [ ]:
stem = "sub-L011/sub-L011_ses-RWTautowait2-20221004-095612"
for v in (EARLIER_VERSION, M.VERSION):
    print(v, [a["path"] for a in list_version(v, stem)])
key = files.path.str.replace(r"(-x)?\.nwb$", "", regex=True)
print("files in this version:", len(files), "| names ending in -x:", files.path.str.endswith("-x.nwb").sum(),
      "| sessions (name without -x) held twice:", key.duplicated().sum())

**Result:** the "-x" copy is in version 0.260825.1909 and not in 0.261120.1450, which holds the session once; no other session is held twice.

## Finding 4

Earlier statement: "The processed trial table has no catch probability. The conversion maps the A struct field under the misspelled name prob_cacth, so prob_catch is dropped; the value is only in the raw trial table (catch_percentage), which has the same rows."

The cell counts the files whose processed trial table has a catch probability column.

In [ ]:
has = files.proc_cols.map(lambda c: "catch_percentage" in c)
odd = files.proc_cols.map(lambda c: any("prob" in x for x in c))
print("files with processed_trials/catch_percentage:", has.sum(), "of", len(files))
print("files with a column named prob_*:", odd.sum())

**Result:** the processed trial table of every file now has the catch probability, as `catch_percentage`.

## Finding 5

Earlier statement: "The raw trial table's is_catch differs from the processed is_catch on 176,009 trials; the processed flag is reward_delay == 100, as the lab's parser defines it, and the raw flag is set on trials with ordinary delays. Neither description says which to use."

The cell counts the trials on which the two flags differ, and prints the descriptions the files give them.

In [ ]:
print("trials on which raw and processed is_catch differ:", files.loc[files.catch_differ >= 0, "catch_differ"].sum(),
      "| files with tables of different lengths:", (files.catch_differ < 0).sum())
for col in ["proc_catch_desc", "raw_catch_desc"]:
    print(col, files[col].value_counts().to_dict())

**Result:** the flags still differ, and the descriptions now say what each is and that they can differ (the processed one marks trials with a 100 s reward delay, the raw one is the Bpod setting).

## Finding 6

Earlier statement: "The subject age is empty in 21,853 of 23,034 files and is the range P6M/P24M in the others, so no rat's age is given."

The cell counts the age and date of birth fields in every file of this version, and in the first file of each rat in the earlier version.

In [ ]:
def ages(df):
    return {"files": len(df), "age given": int((df.age != "").sum()),
            "age values": df.age[df.age != ""].value_counts().head(5).to_dict(),
            "date of birth given": int((df.date_of_birth != "").sum()),
            "rats with a date of birth": df.loc[df.date_of_birth != "", "subject"].nunique()}

print(M.VERSION, ages(files))
earlier = pd.DataFrame(list_version(EARLIER_VERSION))
earlier["subject"] = earlier.path.str.split("/").str[0]
first = earlier.sort_values("path").drop_duplicates("subject")
with ThreadPoolExecutor(24) as ex:
    early = pd.DataFrame(list(ex.map(read_subject, first.to_dict("records"))))
print(EARLIER_VERSION, "(first file of each rat)", ages(early))
dob = files[files.date_of_birth != ""].groupby("subject").date_of_birth.nunique()
print("rats with more than one date of birth in this version:", (dob > 1).sum())

**Result:** the age field is as it was: empty in the 21,852 files that give a date of birth (275 rats), and the range P6M/P24M in the 1,181 files of the 16 rats that give none. The earlier version shows the same split in the first file of each rat, so the date of birth was there before. The age of the 16 rats is still not given.

## Finding 9

Earlier statement: "Equation 6 lets a mixed block become a low (or high) block with probability H0, while every model script uses H0/2 for each (p_b in GenerateSynthData_Bayes.m). Question for the authors - which was used for the fits? Both give the same outcome for the Bayes observer claim here."

The Methods' Equation 6 reads, in the paper's text, P(B_t = Low | B_t-1) = 1 - H0 for Low, H0 for Mixed and 0 for High (and so for High), with H0 = 1/40. The cell prints the transition matrix of the wait time model script at the pinned commit.

In [ ]:
src = M._get(CODE + "BehavioralModel/WaitTimeModels/GenerateSynthData_Bayes.m").decode()
lines = src.splitlines()
i = next(k for k, l in enumerate(lines) if l.strip().startswith("p_b ="))
print("\n".join(lines[i:i + 3]))
print(next(l for l in lines if l.strip().startswith("H0 =")))

**Result:** the script's mixed-to-low and mixed-to-high entries are H0/2 each, while Equation 6 gives H0. The paper and its code still differ.

## Finding 10

Earlier statement: "Figure5.m lists 16 rats but 20 start dates, and the Methods' 30-session offset for switched rats is not in it; the code uses the first 16 dates. Question for the authors - are the dates paired with the rats in order, and were sessions before the 30th pre-cue session left out?"

The cell counts the rats and dates in Figure5.m and looks for the offset in it. It then compares each rat's date, paired in order, with the first session in this version whose task argument `center_port_cue` is true.

In [ ]:
src = M._get(CODE + "Figures/Figure5.m").decode()
block = lambda name: src.split(name + " =", 1)[1].split(";", 1)[0]
rats5 = re.findall(r"'([A-Z]\d{3})'", block("ratListAll"))
dates5 = re.findall(r"'(\d{4}-\d{2}-\d{2})'", block("startDates"))
print("rats:", len(rats5), "| start dates:", len(dates5))
print("lines that use the start dates:", [l.strip() for l in src.splitlines() if "startDates(" in l])
print("lines with 30:", [l.strip() for l in src.splitlines() if re.search(r"\b30\b", l)])
cue = files[files.center_port_cue.str.lower() == "true"]
first_cue = cue.groupby("subject").date.min()
first_any = files.groupby("subject").date.min()
cmp = pd.DataFrame({"paired date": [d.replace("-", "") for d in dates5[:16]],
                    "first cue session": first_cue.reindex(rats5).to_numpy(),
                    "first session": first_any.reindex(rats5).to_numpy()}, index=rats5)
print(cmp.to_string())
print("rats with flagged sessions:", cue.subject.nunique(), "| all among the 16:", set(cue.subject) <= set(rats5))

**Result:** Figure5.m still lists 16 rats and 20 dates, uses `startDates(rr)` for rat `rr`, and has no 30-session offset. With the dates paired in order, the first flagged session of 9 of the 12 rats that have earlier sessions is on the paired date or the day after; for E005, S042 and S050 it is 14, 34 and 41 days later.

## Finding 12

Earlier statement: "myTauShuffleTest.m refits the shuffled groups' mean coefficients times 10 with D at most 1, and the observed groups' coefficients unscaled; for wait times the coefficients include the current offer (about 0.2, so about 2 after scaling), and every shuffled fit sits at the bound, so the null is not on the observed scale. The Methods describe the fits on previous trial coefficients only. Question for the authors - was the wait time test meant to include the current offer?"

The cell prints the fits of myTauShuffleTest.m and the design matrix of regress_wt_vs_rew.m at the pinned commit. It then fits, per rat, the regression of regress_wt_vs_rew.m (z-scored wait times of opt-out catch trials in mixed blocks on the current and 9 previous log2 offers, set to NaN outside mixed blocks, and a constant) and prints the mean coefficient of the current offer. The regression runs over each rat's sessions in date order, with the offers of one session not carried into the next; the paper's code concatenates sessions, which differs only at session starts.

In [ ]:
shuffle = M._get(CODE + "Analysis/myTauShuffleTest.m").decode().splitlines()
print("\n".join(l for l in shuffle if "fit_exp_decay" in l or "[-" in l))
reg = M._get(CODE + "Analysis/regress_wt_vs_rew.m").decode().splitlines()
print("\n".join(l for l in reg if "x(:,j)" in l or "R = log2" in l or "usethese =" in l))

NBACK = 10
order = files.sort_values(["subject", "date", "path"]).index
coef = {}
for rat, idx in files.loc[order].groupby("subject").groups.items():
    rows = []
    for k in idx:
        a = res[k][1]
        vol = pd.Series(a["vol"]).map(M.VOL_MAP).to_numpy(float)
        R = np.where(a["block"] == b"Mixed", np.log2(vol), np.nan)
        X = np.column_stack([np.r_[np.zeros(j), R[:len(R) - j]] for j in range(NBACK)])
        use = a["is_opt_out"] & a["is_catch"] & (a["block"] == b"Mixed") & np.isfinite(a["wait_time"])
        rows.append((X[use], a["wait_time"][use]))
    X = np.vstack([r[0] for r in rows])
    y = np.concatenate([r[1] for r in rows])
    ok = np.isfinite(X).all(1)
    X, y = X[ok], y[ok]
    if len(y) <= NBACK + 1:
        continue
    y = (y - y.mean()) / y.std(ddof=1)
    coef[rat] = np.linalg.lstsq(np.column_stack([X, np.ones(len(y))]), y, rcond=None)[0]
coef = pd.DataFrame(coef).T
print("rats:", len(coef), "| mean coefficient of the current offer:", round(coef[0].mean(), 3),
      "| times 10:", round(10 * coef[0].mean(), 3), "| median of the previous offer's:", round(coef[1].median(), 4))

**Result:** the shuffled groups' coefficients are still fitted times 10 with the amplitude bounded at 1 (`[-3 0], [1 50]`), the observed groups' unscaled within `[-10 0], [1 10]`, and the wait time design matrix still starts at the current offer (`j = 1` takes `R` unshifted). The current offer's coefficient, times 10, is above the bound of 1, as the earlier finding said. The fits of the shuffled groups themselves were not re-run here.

## Summary

| Earlier finding | On version 0.261120.1450 |
|---|---|
| 3, session deposited twice | Fixed: the "-x" copy is gone and no session is held twice. |
| 4, no catch probability in the processed table | Fixed: the processed table has `catch_percentage`. |
| 5, raw and processed is_catch differ, undocumented | Fixed: the flags still differ by design, and both descriptions now say how. |
| 6, no age | Still present for the 16 rats without a date of birth; the other 275 have one, as in the earlier version. |
| 9, Equation 6 against H0/2 in the code | Still present (the code is unchanged). |
| 10, 16 rats and 20 dates in Figure5.m | Still present (the code is unchanged). |
| 12, scaling in myTauShuffleTest.m | Still present (the code is unchanged). |

In [ ]:
print(f"total run time {time.time() - T0:.0f} s")